# 139 · Document Vision Agent

Render PDF pages as images, pass them to a vision LLM, and extract structured data page-by-page using a LangGraph workflow.

**What you'll learn:**
- How to render PDF pages to PNG using `pypdfium2` and tune DPI via the `scale` parameter
- How to encode rendered pages as base64 for the OpenAI vision API
- How to write a JSON-schema extraction prompt that produces reliable structured output
- How the map-reduce pattern (extract per page → combine results) enables parallelism
- How to estimate token cost before running extraction on a full document
- How the two-node LangGraph `DocState` workflow (load → extract) is wired together

**Source:** `examples/139-doc-vision-agent/`

**Requirements:**
- Part A: no API key needed — all cells run on CPU with Pillow only
- Part B: `OPENAI_API_KEY` in your environment (or `.env` file) + `pypdfium2` installed

In [ ]:
%pip install -q openai httpx python-dotenv langgraph pypdfium2 Pillow

---
## Part A — CPU-Safe Concept Demonstrations

All cells in Part A run without an API key and without `pypdfium2`. They use only `Pillow` and plain Python to illustrate every concept in the pipeline.

### Concept: PDF Rendering Library Landscape

Several Python libraries can render PDF pages to images. Each makes different tradeoffs:

| Library | Approach | Pros | Cons | Best for |
|---|---|---|---|---|
| `pypdfium2` | PDFium C library via Python bindings | Fast, accurate, handles complex PDFs | Binary dep, no text layer | Production rendering |
| `pdf2image` | Poppler via subprocess | Easy install, good quality | Requires system Poppler | CI / simpler setups |
| `pdfplumber` | pdfminer.six wrapper | Extracts text, tables, coordinates | No image output | Text / table extraction |
| `PyMuPDF (fitz)` | MuPDF engine | Very fast, many output formats | GPL license | High-perf rendering |

**When to use vision extraction vs text extraction:**

- **Text extraction** (pdfplumber / pdfminer) is cheaper and faster when the PDF is text-selectable and you only need prose or tabular data.
- **Vision extraction** is necessary for scanned PDFs, image-heavy layouts, forms with unusual structure, or whenever spatial context (charts, diagrams, decorative callouts) carries meaning that raw text loses.
- In practice, combine both: extract text programmatically where possible, and fall back to vision for pages where text extraction yields garbage or nothing.

### Concept: Page Selection Strategies

Processing every page of a long document is expensive. Four strategies for selecting which pages to send to the vision model:

1. **First N pages** — Good for executive summaries, title pages, and abstracts. Fast and predictable.
2. **All pages** — Thorough but costs scale linearly. Only sensible for short documents (≤10 pages).
3. **Keyword-guided selection** — Run a fast OCR or text-extraction pass first, pick only pages containing target keywords. Cheapest for large docs.
4. **Thumbnail-first selection** — Render at low DPI (scale=1), send thumbnails to the vision model with a "which pages are relevant?" prompt, then re-render selected pages at high DPI.

**Decision tree:**

```
Is the PDF text-selectable?
├── YES → Use pdfplumber for text; use vision only for tables/charts
└── NO  → Must use vision rendering
    ├── Small doc (≤10 pages) → render all pages
    └── Large doc (>10 pages) → keyword-guided or thumbnail-first
```

The workflow in this example (`load → extract`) always processes all pages up to `MAX_PAGES`. Exercise 2 shows how to add a page selector.

### Concept: DPI vs Token Cost Tradeoffs

`pypdfium2` controls output resolution with a `scale` parameter. Scale maps to effective DPI as follows (PDFs are natively 72 DPI):

| Scale | Effective DPI | PNG size (typical A4) | Vision tokens (est.) | Cost/page (gpt-4o-mini) |
|---|---|---|---|---|
| 1 (low) | 72 | ~100 KB | ~255 | $0.00026 |
| 2 (default) | 150 | ~350 KB | ~765 | $0.00078 |
| 4 (high) | 300 | ~1.2 MB | ~2295 | $0.00234 |

**Rule of thumb:** use `scale=2` for most production work. The quality is sufficient for normal body text, tables, and diagrams. Use `scale=4` only when you need to read very small print, complex equations, or dense technical diagrams.

Token estimates above follow OpenAI's tile-based image token counting: images are divided into 512×512 tiles, each costing 170 tokens, plus an 85-token base. A typical A4 page at scale=2 produces roughly 1240×1754 pixels → 3×4 tiles → 765 tokens.

### Concept: Extract → Combine (Map-Reduce Pattern)

The workflow uses a map-reduce pattern. Each page is an independent unit of work:

```
PDF bytes
    │
    ▼
[page_to_b64] × N pages  ←── map phase (render)
    │
    ▼
[extract_page LLM] × N  ←── map phase (vision extract, parallel-safe)
    │
    ▼
[combine results]        ←── reduce phase (aggregate)
    │
    ▼
Unified findings list
```

Because each page extraction is independent, the map phase is trivially parallelisable. The current `extract_node` in `workflow.py` runs pages sequentially. To parallelise, replace the loop with `asyncio.gather()` or a `ThreadPoolExecutor` — no changes to the graph topology are needed.

The reduce phase aggregates: total issue counts, average scores, deduplicated term lists, or any cross-page synthesis your use case requires.

In [ ]:
from PIL import Image, ImageDraw
import io, base64


def create_mock_page(text: str, width: int = 600, height: int = 800) -> Image.Image:
    """Create a synthetic document page image for concept demonstrations."""
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([20, 20, width - 20, 60], fill="#2563EB")
    draw.text((30, 33), "WCAG Accessibility Report", fill="white")
    y = 80
    for line in text.split("\n"):
        draw.text((30, y), line, fill="black")
        y += 22
    draw.line([20, height - 30, width - 20, height - 30], fill="#CBD5E1", width=1)
    draw.text((30, height - 22), "Confidential — Draft v0.3", fill="#94A3B8")
    return img


mock_text = (
    "Section 1.1: Non-text Content\n"
    "All images must have alt text.\n"
    "Failing elements: 3\n"
    "Page score: 72/100"
)
page_img = create_mock_page(mock_text)

# Encode to base64 — same operation as pdf_page_to_b64() in tools.py
buf = io.BytesIO()
page_img.save(buf, format="PNG")
b64_page = base64.b64encode(buf.getvalue()).decode()

print(f"Mock page created: {page_img.size[0]}x{page_img.size[1]} px")
print(f"PNG bytes        : {len(buf.getvalue()):,}")
print(f"Base64 length    : {len(b64_page):,} chars")
print(f"Base64 preview   : {b64_page[:60]}...")

In [ ]:
import json

# Simulate what extract_page() returns for 3 pages of an accessibility report
MOCK_EXTRACTIONS = [
    {"page": 1, "section": "1.1 Non-text Content",  "issues": 3, "score": 72},
    {"page": 2, "section": "1.3 Adaptable Content", "issues": 1, "score": 91},
    {"page": 3, "section": "1.4 Distinguishable",   "issues": 7, "score": 58},
]


def mock_extract_page(page_num: int) -> dict:
    """Simulate an LLM extraction result without calling the API."""
    return MOCK_EXTRACTIONS[page_num - 1]


# Map phase — one extraction per page
all_results = [mock_extract_page(i) for i in range(1, 4)]

# Reduce phase — aggregate across pages
total_issues = sum(r["issues"] for r in all_results)
avg_score    = sum(r["score"]  for r in all_results) / len(all_results)

print("=== Combined Extraction Results ===")
for r in all_results:
    print(f"  Page {r['page']}: {r['section']:<30} {r['issues']} issues, score {r['score']}")
print(f"\nTotal issues : {total_issues}")
print(f"Average score: {avg_score:.1f}/100")

In [ ]:
# Token and cost estimation for PDF vision extraction
# Pricing as of mid-2024 — verify at https://openai.com/pricing

GPT4O_MINI_INPUT_PER_1M  = 0.15   # USD per 1M input tokens
GPT4O_MINI_OUTPUT_PER_1M = 0.60   # USD per 1M output tokens


def estimate_extraction_cost(
    num_pages: int,
    scale: int = 2,
    output_tokens_per_page: int = 512,
) -> dict:
    """Estimate API cost before running extraction on a full document."""
    tokens_per_scale = {1: 255, 2: 765, 4: 2295}
    img_tokens    = tokens_per_scale.get(scale, 765)
    prompt_tokens = 200  # schema_prompt overhead

    total_input  = (img_tokens + prompt_tokens) * num_pages
    total_output = output_tokens_per_page * num_pages
    input_cost   = (total_input  / 1_000_000) * GPT4O_MINI_INPUT_PER_1M
    output_cost  = (total_output / 1_000_000) * GPT4O_MINI_OUTPUT_PER_1M

    return {
        "pages":           num_pages,
        "scale":           scale,
        "input_tokens":    total_input,
        "output_tokens":   total_output,
        "input_cost_usd":  round(input_cost,               5),
        "output_cost_usd": round(output_cost,              5),
        "total_cost_usd":  round(input_cost + output_cost, 5),
    }


print(f"{'Pages':>6} | {'Scale':>5} | {'Input tok':>10} | {'Output tok':>10} | {'Cost USD':>10}")
print("-" * 56)
for pages in [3, 10, 50, 100]:
    for scale in [2, 4]:
        est = estimate_extraction_cost(pages, scale)
        print(
            f"{est['pages']:>6} | {est['scale']:>5} | "
            f"{est['input_tokens']:>10,} | {est['output_tokens']:>10,} | "
            f"${est['total_cost_usd']:>9.4f}"
        )

---
## Exercise 1: Switch to JSON Schema Output

The current `extract_page()` in `tools.py` appends `"\n\nRespond with valid JSON only."` to whatever `schema_prompt` you pass in. This is fine for getting *some* JSON back, but it doesn't enforce a specific structure.

**Your task:** Write a `schema_prompt` that instructs the model to return a JSON object with exactly these keys:

| Key | Type | Description |
|---|---|---|
| `section_title` | `str` | The main heading visible on this page |
| `key_findings` | `list[str]` | Main findings, max 3 items |
| `compliance_score` | `int` | Quality or compliance level, 0–100 |
| `action_items` | `list[str]` | Recommended fixes, max 3 items |

**Tips:**
- Include a concrete JSON example — models follow examples more reliably than prose descriptions.
- Add explicit rules about list length limits.
- End with `"Respond with ONLY the JSON object, no markdown fences, no extra text."` to make parsing reliable.

In [ ]:
# Exercise 1: Write a JSON-schema extraction prompt
# ---------------------------------------------------------------------------
# TODO: Write a schema_prompt that asks the model to return a JSON object
#       with exactly these keys:
#         - section_title    (str)
#         - key_findings     (list[str], max 3 items)
#         - compliance_score (int, 0-100)
#         - action_items     (list[str], max 3 items)
#
# Include a concrete JSON example so the model knows the exact structure.
# ---------------------------------------------------------------------------

schema_prompt = """
# TODO: fill in your prompt here
"""

print(schema_prompt)

In [ ]:
# Answer Key — Exercise 1
# ---------------------------------------------------------------------------
schema_prompt = """Analyze this document page and return a JSON object with exactly these keys:
{
  "section_title": "the main heading or section title on this page",
  "key_findings": ["finding 1", "finding 2", "finding 3"],
  "compliance_score": 85,
  "action_items": ["fix 1", "fix 2", "fix 3"]
}

Rules:
- section_title: the primary heading visible on this page
- key_findings: list of up to 3 observations about this page's content
- compliance_score: integer 0-100 reflecting accessibility or quality level
- action_items: up to 3 concrete, specific improvement suggestions
- Respond with ONLY the JSON object, no markdown fences, no extra text"""

print("Schema prompt ready:")
print(schema_prompt)
print()
# Key insight: schema_prompt is completely separate from the page image.
# The same prompt can be reused across different documents — just swap the image.
# extract_page() in tools.py appends 'Respond with valid JSON only.' automatically.

---
## Exercise 2: Select Only Odd Pages

The `extract_node` in `workflow.py` currently processes all pages with:

```python
for i in range(state["page_count"]):
    ...
```

**Your task:** Write a `filtered_extract` function that accepts a `page_indices` list and only processes the specified pages. Then demonstrate it by extracting only odd-numbered pages (1, 3, 5, …) from the mock dataset.

This pattern is the foundation for all page-selection strategies described earlier.

In [ ]:
# Exercise 2: Select only odd pages
# ---------------------------------------------------------------------------
# Assume page_count = 6 and that mock_extract_page(n) returns a result for
# pages 1-6 (extend MOCK_EXTRACTIONS or use a lambda for this exercise).
#
# TODO: implement filtered_extract(page_count, page_indices, extractor_fn)
#   - page_count:   total pages in the document
#   - page_indices: list of 0-based page indices to process
#   - extractor_fn: callable(page_num: int) -> dict
#
# TODO: call it with odd page indices for a 6-page document
#   (pages 1, 3, 5 → 0-based indices 0, 2, 4)
# ---------------------------------------------------------------------------

def filtered_extract(page_count, page_indices, extractor_fn):
    # TODO: implement
    pass


# TODO: call filtered_extract and print results


In [ ]:
# Answer Key — Exercise 2
# ---------------------------------------------------------------------------

MOCK_6_PAGES = [
    {"page": 1, "section": "1.1 Non-text Content",    "issues": 3,  "score": 72},
    {"page": 2, "section": "1.2 Time-based Media",    "issues": 0,  "score": 100},
    {"page": 3, "section": "1.3 Adaptable Content",   "issues": 1,  "score": 91},
    {"page": 4, "section": "1.4 Distinguishable",     "issues": 7,  "score": 58},
    {"page": 5, "section": "2.1 Keyboard Accessible", "issues": 2,  "score": 83},
    {"page": 6, "section": "2.2 Enough Time",         "issues": 0,  "score": 100},
]


def filtered_extract(page_count: int, page_indices: list, extractor_fn) -> list:
    """Run extraction only on the requested page indices (0-based)."""
    results = []
    for idx in page_indices:
        if 0 <= idx < page_count:
            result = extractor_fn(idx + 1)  # extractor_fn takes 1-based page num
            results.append(result)
    return results


# Select odd pages: 0-based indices 0, 2, 4 → pages 1, 3, 5
page_count  = 6
odd_indices = [i for i in range(page_count) if i % 2 == 0]  # 0, 2, 4
mock_fn     = lambda page_num: MOCK_6_PAGES[page_num - 1]

odd_results = filtered_extract(page_count, odd_indices, mock_fn)

print("Extracted odd pages (1, 3, 5):")
for r in odd_results:
    print(f"  Page {r['page']}: {r['section']:<35} score {r['score']}")
print(f"\nPages skipped: {page_count - len(odd_results)} (even pages not sent to LLM)")

### Concept: LangGraph Workflow Walk-through

The `create_workflow()` function in `workflow.py` builds a two-node LangGraph graph:

```
START → load_node → extract_node → END
```

`DocState` is a `TypedDict` with these fields:

| Field | Set by | Description |
|---|---|---|
| `pdf_source` | caller | URL or local path to the PDF |
| `schema_prompt` | caller | JSON schema extraction prompt |
| `page_count` | `load_node` | Total pages in the loaded PDF |
| `results` | `extract_node` | List of per-page extraction dicts |

Runtime configuration (OpenAI client, model name) is passed through `config["configurable"]` rather than through state. This keeps the LLM client out of the serialisable state graph, which matters for checkpointing and replay.

In [ ]:
# Demonstrate DocState construction without running the graph
from typing import TypedDict


class DocState(TypedDict, total=False):
    pdf_source:    str
    schema_prompt: str
    page_count:    int
    results:       list


# This is the initial state you would pass to graph.invoke()
initial_state: DocState = {
    "pdf_source":    "https://arxiv.org/pdf/1706.03762",
    "schema_prompt": schema_prompt,  # from Exercise 1 answer
    "page_count":    0,
    "results":       [],
}

print("Initial DocState:")
for k, v in initial_state.items():
    display_v = v if not isinstance(v, str) or len(v) < 60 else v[:57] + "..."
    print(f"  {k:<16} : {display_v!r}")

print()
print("After load_node   : page_count is set to the actual PDF page count.")
print("After extract_node: results contains one dict per processed page.")

### Concept: Robust JSON Parsing

Even when you ask the model to return "ONLY valid JSON", it sometimes wraps the response in markdown code fences. The `extract_page()` function handles this with:

```python
text = resp.choices[0].message.content.strip().strip("```json").strip("```")
try:
    return json.loads(text)
except json.JSONDecodeError:
    return {"raw": text}  # graceful fallback
```

The fallback `{"raw": text}` means a parse failure never crashes the pipeline — you get a degraded result for that page, and all other pages succeed. In production you might want to:
1. Retry the extraction once before falling back
2. Use `response_format={"type": "json_object"}` (gpt-4o and gpt-4o-mini support this) to force valid JSON from the model side
3. Log `{"raw": ...}` entries for manual review

In [ ]:
import json


def safe_parse_llm_json(text: str) -> dict:
    """Strip markdown fences and parse JSON, with graceful fallback."""
    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"raw": text}


# Simulate the three response forms you might get from a vision model
test_cases = [
    ("clean JSON",           '{"section_title": "Results", "compliance_score": 88}'),
    ("markdown-fenced JSON", '```json\n{"section_title": "Results", "compliance_score": 88}\n```'),
    ("plain code fence",     '```\n{"section_title": "Results", "compliance_score": 88}\n```'),
    ("broken JSON",          '{"section_title": "Results" compliance_score: 88}'),
]

for label, raw in test_cases:
    result = safe_parse_llm_json(raw)
    status = "OK" if "raw" not in result else "FALLBACK"
    print(f"[{status:>8}] {label}: {result}")

---
## Part B — Live PDF Extraction

**Requirements:**
- `OPENAI_API_KEY` set in your environment or in a `.env` file
- `pypdfium2` installed (`%pip install -q pypdfium2`)
- Internet access to download the PDF

**Cost estimate:** 3 pages × scale=2 ≈ 3 × $0.00078 ≈ **$0.003 per run** (less than half a cent).

All cells below skip gracefully if the requirements are not met.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

_READY = True

if not os.environ.get("OPENAI_API_KEY"):
    print("SKIP: OPENAI_API_KEY not set. Add it to your .env or shell to run Part B.")
    _READY = False
else:
    print("OK: OPENAI_API_KEY found")

try:
    import pypdfium2  # noqa: F401
    print("OK: pypdfium2 available")
except ImportError:
    print("SKIP: pypdfium2 not installed. Run: pip install pypdfium2")
    _READY = False

if _READY:
    print("\nReady for Part B.")

In [ ]:
if not _READY:
    print("Skipping — requirements not met (see setup cell above).")
else:
    import httpx
    import pypdfium2 as pdfium
    import io, base64

    # Attention Is All You Need — small, always-available public PDF
    PDF_URL   = "https://arxiv.org/pdf/1706.03762"
    MAX_PAGES = 3

    print(f"Downloading PDF from {PDF_URL} ...")
    pdf_bytes = httpx.get(PDF_URL, timeout=30, follow_redirects=True).content
    print(f"Downloaded: {len(pdf_bytes):,} bytes")

    doc = pdfium.PdfDocument(pdf_bytes)
    total_pages     = len(doc)
    pages_to_render = min(MAX_PAGES, total_pages)
    print(f"Total pages in PDF: {total_pages}, rendering first {pages_to_render}")

    page_b64s = []
    for i in range(pages_to_render):
        page   = doc[i]
        bitmap = page.render(scale=2)
        img    = bitmap.to_pil()
        buf    = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        page_b64s.append(b64)
        print(f"  Page {i+1}: {img.size[0]}x{img.size[1]} px, {len(b64):,} b64 chars")

    print(f"\nAll {pages_to_render} pages rendered and encoded.")

In [ ]:
if not _READY:
    print("Skipping — requirements not met.")
else:
    import openai

    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    SCHEMA_PROMPT = """Analyze this research paper page and return JSON only:
    {
      "section_title": "the heading visible on this page, or 'Cover' for title pages",
      "key_findings": ["finding 1", "finding 2"],
      "technical_terms": ["term 1", "term 2", "term 3"]
    }
    Respond with ONLY valid JSON, no markdown fences."""

    results = []
    for i, b64 in enumerate(page_b64s):
        print(f"Extracting page {i + 1} / {len(page_b64s)} ...")
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": [
                {"type": "image_url",  "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text",       "text": SCHEMA_PROMPT},
            ]}],
            max_tokens=512,
        )
        raw  = resp.choices[0].message.content.strip()
        data = safe_parse_llm_json(raw)
        data["page"] = i + 1
        results.append(data)
        print(f"  -> {data.get('section_title', '(no title)')}")

    print("\n=== All Extractions ===")
    for r in results:
        print(f"\nPage {r['page']}: {r.get('section_title', 'N/A')}")
        for f in r.get("key_findings", []):
            print(f"  - {f}")
        terms = r.get("technical_terms", [])
        if terms:
            print(f"  Terms: {', '.join(terms)}")

In [ ]:
if not _READY:
    print("Skipping — requirements not met.")
else:
    # Reduce phase: aggregate all findings across pages
    all_terms    = []
    all_findings = []
    for r in results:
        all_terms.extend(r.get("technical_terms", []))
        all_findings.extend(r.get("key_findings",  []))

    unique_terms = sorted(set(all_terms), key=str.lower)

    print(f"Pages processed : {len(results)}")
    print(f"Total findings  : {len(all_findings)}")
    print(f"Unique terms    : {len(unique_terms)}")
    print()
    print("All unique technical terms:")
    for t in unique_terms:
        print(f"  - {t}")

---
## Exercise 3: Add a Confidence Score

Modify the `SCHEMA_PROMPT` (from Part B) to also request a `confidence` field (float 0.0–1.0) where the model rates its own certainty that it correctly identified the section title.

Self-reported confidence is surprisingly well-calibrated in modern vision models and can be used to flag pages for human review.

**Your task:** Update the JSON schema and test `safe_parse_llm_json` on a hardcoded response string that includes the new field.

In [ ]:
# Exercise 3: Add confidence field to the extraction schema
# ---------------------------------------------------------------------------
# TODO: Write an updated schema_prompt that adds:
#   "confidence": 0.9   (float 0.0-1.0, model's self-reported certainty)
#
# Include a description of what 1.0 vs 0.5 vs 0.2 means to guide calibration.
# ---------------------------------------------------------------------------

updated_schema_prompt = """
# TODO: fill in your updated schema prompt here
"""

print(updated_schema_prompt)

In [ ]:
# Answer Key — Exercise 3
# ---------------------------------------------------------------------------
updated_schema_prompt = """Analyze this research paper page and return JSON only:
{
  "section_title": "the heading visible on this page, or 'Cover' for title pages",
  "key_findings": ["finding 1", "finding 2"],
  "technical_terms": ["term 1", "term 2", "term 3"],
  "confidence": 0.9
}

Rules:
- confidence: your certainty that section_title is correct, float 0.0-1.0
- 1.0 = very clear heading; 0.5 = ambiguous layout; 0.2 = dense equations or figures
- Respond with ONLY valid JSON, no markdown fences."""

print("Updated schema prompt ready.")

# Test parse of the new schema shape without an API call
mock_response = json.dumps({
    "section_title":  "Attention Mechanism",
    "key_findings":   ["Scaled dot-product attention"],
    "technical_terms": ["softmax", "query", "key", "value"],
    "confidence":     0.95,
})

parsed = safe_parse_llm_json(mock_response)
print("\nMock parse result:")
for k, v in parsed.items():
    print(f"  {k}: {v}")

# Pages with low confidence are good candidates for human review
confidence = parsed.get("confidence", 1.0)
flag = " << FLAG FOR REVIEW" if confidence < 0.6 else ""
print(f"\nConfidence: {confidence}{flag}")

---
## What's Next

**Go deeper on document vision:**

- **NOUGAT** (Meta, 2023): "Neural Optical Understanding for Academic Documents" — a transformer trained specifically to parse academic PDFs into structured Markdown, including LaTeX equations. arxiv:[2308.13418](https://arxiv.org/abs/2308.13418). Good baseline to compare against a general vision model.

- **ColPali** (2024): Multi-vector retrieval directly over PDF page images without any OCR or text extraction step. Embeds the page image itself and retrieves relevant pages at query time. arxiv:[2407.01449](https://arxiv.org/abs/2407.01449). The logical next step when you need retrieval over large document corpora.

- **Example 138 — vision-qa-agent**: Single-image Q&A as a simpler starting point. If your use case is one image at a time rather than multi-page documents, start there.

- **Example 42 — multimodal-rag**: Embedding image descriptions alongside text chunks for hybrid retrieval. Pairs naturally with this example: use doc-vision-agent to extract descriptions, then index them with a retrieval workflow.

- **`response_format={"type": "json_object"}`**: Available on `gpt-4o` and `gpt-4o-mini`. Forces the model to return valid JSON without fence-stripping. Drop this into `extract_page()` in `tools.py` to make JSON parsing rock-solid for production use.